# RSO-901 Glycol Set Points — Date Range

Chiller glycol **set point** changes over an arbitrary `day_obs` range, from the
`lsst.sal.HVAC.logevent_chillerConfiguration` event. Set the `start_day_obs` and `end_day_obs`
parameters below (both integers `YYYYMMDD`); the window runs from 12:00 UTC on the first `day_obs`
through 11:59:59 UTC the day after the last one. All four HVAC chillers are drawn as **step** lines on one shared,
zoomable panel — a set point is piecewise-constant, so the last value is held to the end of the
window. Genuine changes are also listed in a change-log table.

Known **catastrophic cooling events** (glycol / Dynalene / PCS faults and near-misses, from
`glycol_catastrophic_faults.csv`) are overlaid as vertical **dashed** lines — red for realised faults,
amber for averted near-misses — so set-point behaviour can be read against known incidents.

See `ai_notes.md` in this folder for the topic survey and field reference. This is the range-scoped
sibling of the *Single Day* and *Monthly* notebooks.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import pandas as pd

from bokeh.io import output_notebook, output_file, save, show
from bokeh.models import ColumnDataSource, HoverTool, Label, Range1d, Span
from bokeh.plotting import figure

from lsst.summit.utils.efdUtils import (
    getEfdData,
    makeEfdClient,
    getDayObsStartTime,
    getDayObsEndTime,
)

output_notebook()

## Parameters

In [ ]:
# Inclusive day_obs range, as integers YYYYMMDD. The observatory day rolls over at
# 12:00 UTC (UTC-12 convention), so the window is 12:00 UTC on start_day_obs through
# 11:59:59 UTC the day after end_day_obs.
start_day_obs = 20260101
end_day_obs = 20260927  # last complete night; RSO-901 covers all of 2026

TOPIC = "lsst.sal.HVAC.logevent_chillerConfiguration"

# Curated CSV of catastrophic glycol / camera-cooling events (faults & near-misses),
# overlaid on the set-point plot as vertical dashed lines.
EVENTS_CSV = "glycol_catastrophic_faults.csv"

# device_id (DeviceId_chiller enum) -> (name, colour).
# Bright, saturated palette so the four step lines stay distinct on a shared panel.
CHILLERS = {
    101: ("chiller01", "#00C2C2"),  # teal
    102: ("chiller02", "#FF7A00"),  # orange
    103: ("chiller03", "#B026D9"),  # purple
    104: ("chiller04", "#1E90FF"),  # blue
}

# Vertical-line colour by event outcome: realised fault vs averted near-miss.
EVENT_COLORS = {"actual": "#E5484D", "near_miss": "#F5A623"}

In [ ]:
if end_day_obs < start_day_obs:
    raise ValueError(f"end_day_obs {end_day_obs} < start_day_obs {start_day_obs}")

begin = getDayObsStartTime(start_day_obs)  # 12:00 UTC on start_day_obs
end = getDayObsEndTime(end_day_obs)  # 11:59:59 UTC the day after end_day_obs

print(f"day_obs {start_day_obs}\u2013{end_day_obs}: {begin.isot} -> {end.isot} (UTC)")

## Query the set-point (re)configuration events

This is a sparse event — a row is written only when a chiller's set point is (re)configured, so even a
multi-month range is cheap to query in one shot.

In [ ]:
df = getEfdData(efd_client := makeEfdClient(), TOPIC, begin=begin, end=end)
print(f"{len(df)} configuration event(s) for day_obs {start_day_obs}\u2013{end_day_obs}")
df[["device_id", "activeSetpoint"]].head() if len(df) else df

## Catastrophic cooling events

Load the curated catalog of glycol / Dynalene / PCS cooling incidents (realised **faults** and averted
**near-misses**) and keep only those inside the plotted window. They are overlaid on the set-point
figure as vertical **dashed** lines — red for realised faults (`outcome == "actual"`), amber for
near-misses — so set-point behaviour can be read against known incidents. Full detail is in the events
table at the bottom.

In [ ]:
events = pd.read_csv(EVENTS_CSV)
# Parse the fault onset as UTC, then drop tz for Bokeh's (UTC) datetime axis.
events["t"] = pd.to_datetime(
    events["fault_start_timestamp_utc"], utc=True
).dt.tz_localize(None)

# Keep only events inside the plotted window.
win_lo, win_hi = begin.utc.datetime, end.utc.datetime
events_win = events[(events["t"] >= win_lo) & (events["t"] <= win_hi)].sort_values("t")

print(f"{len(events_win)} catastrophic event(s) within the window")
(
    events_win[
        ["event_id", "t", "outcome", "consequence", "camera_system", "fracas_ticket"]
    ]
    if len(events_win)
    else "No catastrophic events recorded in this window."
)

## Figure — all chillers on a single panel

All four chillers are drawn as **step** lines on one shared panel, one colour each. A set point is
piecewise-constant, so the last value is held to the end of the window. Use the box-zoom /
wheel-zoom / pan tools to inspect clusters of changes; catastrophic events appear as vertical dashed
lines tagged `#N` (see the events table below).

In [ ]:
# Naive UTC datetimes for Bokeh (its datetime axis assumes UTC).
x_lo = begin.utc.datetime
x_hi = end.utc.datetime


def build_figure():
    """Build the single-panel step plot from fresh Bokeh models.

    Bokeh models can belong to only one document, so `save(...)` (file doc) and
    `show(...)` (notebook doc) each need their own set of objects — hence a builder.
    """
    p = figure(
        height=460,
        sizing_mode="stretch_width",
        x_axis_type="datetime",
        x_range=Range1d(start=x_lo, end=x_hi),
        title=f"HVAC chiller active set points — day_obs {start_day_obs}\u2013{end_day_obs}",
        tools="xpan,xwheel_zoom,box_zoom,reset,save",
        active_scroll="xwheel_zoom",
    )

    for dev_id, (name, color) in CHILLERS.items():
        g = df[df["device_id"] == dev_id].sort_index() if len(df) else df
        if not len(g):
            continue
        # Drop tz (values are UTC) and hold the last value to the end of the window.
        gi = g.index
        gi = gi.tz_localize(None) if gi.tz is not None else gi
        xs = list(gi) + [x_hi]
        ys = list(g["activeSetpoint"]) + [g["activeSetpoint"].iloc[-1]]
        src = ColumnDataSource(dict(x=xs, y=ys))
        step = p.step(
            x="x", y="y", source=src, mode="after",
            line_color=color, line_width=2, legend_label=name,
        )
        p.add_tools(
            HoverTool(
                renderers=[step],
                tooltips=[
                    ("chiller", name),
                    ("time", "@x{%F %T}"),
                    ("set point", "@y{0.00} °C"),
                ],
                formatters={"@x": "datetime"},
                mode="mouse",
            )
        )

    # Overlay catastrophic cooling events as vertical dashed lines (fresh models
    # each call — a Span/Label belongs to a single document, like the glyphs above).
    for _, ev in events_win.iterrows():
        color = EVENT_COLORS.get(ev["outcome"], "#888888")
        p.add_layout(
            Span(
                location=ev["t"],
                dimension="height",
                line_color=color,
                line_dash="dashed",
                line_width=2,
                line_alpha=0.9,
            )
        )
        # Pin a small tag near the top (screen-space y) so it survives auto y-ranging.
        p.add_layout(
            Label(
                x=ev["t"],
                y=8,
                y_units="screen",
                text=f"#{int(ev['event_id'])}",
                text_color=color,
                text_font_size="9pt",
                text_align="center",
                background_fill_color="white",
                background_fill_alpha=0.6,
            )
        )

    p.xaxis.axis_label = "Time [UTC]"
    p.yaxis.axis_label = "Set point [°C]"
    p.legend.location = "top_left"
    p.legend.click_policy = "hide"
    return p


os.makedirs("plots", exist_ok=True)
stem = f"chiller_setpoints_{start_day_obs}_{end_day_obs}"
output_file(f"plots/{stem}.html", title=f"Chiller set points {start_day_obs}-{end_day_obs}")
save(build_figure())  # fresh models -> file document

output_notebook()  # switch rendering back to inline
show(build_figure())  # fresh models -> notebook document

## Technote figure

Writes the two-panel interactive figure embedded in SOTN-012 (set point per chiller, plus the number of
genuine set-point changes per week) to `../_extra/plots/`, which Sphinx copies to the published site.
After re-running for a new range, update the `<iframe>` path in `index.md` if the file name changed.

In [ ]:
from setpoint_plots import build_technote_figure

# Drop tz (values are UTC) for Bokeh's datetime axis.
df_naive = df.copy()
if df_naive.index.tz is not None:
    df_naive.index = df_naive.index.tz_localize(None)

technote_html = f"../_extra/plots/{stem}.html"
os.makedirs(os.path.dirname(technote_html), exist_ok=True)
output_file(technote_html, title=f"Chiller set points {start_day_obs}-{end_day_obs}")
save(build_technote_figure(df_naive, events_win, x_lo, x_hi))

output_notebook()
show(build_technote_figure(df_naive, events_win, x_lo, x_hi))

## Change-log table

Keep only rows where the set point actually moved (per chiller). This also drops redundant
re-publications (e.g. a CSC restart re-emits the current config unchanged).

In [ ]:
if len(df):
    changes = (
        df.sort_index()
        .groupby("device_id")["activeSetpoint"]
        .apply(lambda s: s[s.diff().ne(0)])  # first row + genuine changes only
        .reset_index()
        .rename(columns={"level_1": "time"})
    )
    changes["chiller"] = changes["device_id"].map(lambda d: CHILLERS[d][0])
    changes["delta_C"] = changes.groupby("device_id")["activeSetpoint"].diff()
    display(changes[["time", "chiller", "activeSetpoint", "delta_C"]])
else:
    print("No configuration events for this range.")

## Catastrophic events in this window

Full detail for the events drawn as vertical dashed lines above. The `#N` tags on the plot are the
`event_id` in this table. `summary` and `fracas_url` give the incident write-up and tracking ticket.

In [ ]:
if len(events_win):
    with pd.option_context("display.max_colwidth", None):
        display(
            events_win[
                [
                    "event_id",
                    "t",
                    "outcome",
                    "consequence",
                    "camera_system",
                    "fracas_ticket",
                    "fracas_url",
                    "summary",
                ]
            ].set_index("event_id")
        )
else:
    print("No catastrophic events recorded in this window.")